In [107]:
import pandas as pd
import numpy as np



In [108]:
data2025 = pd.read_csv('data/arrests-latest.csv')
columns_to_keep3 = ["apprehension_date", "apprehension_aor", 'duplicate_likely', 'unique_identifier', "apprehension_criminality", 'birth_year','gender','citizenship_country']
data2025 = data2025[columns_to_keep3]
data2025 = data2025.rename(columns={'apprehension_aor': 'aor_nam', 'apprehension_date': 'Apprehension Date', 'apprehension_criminality': 'Apprehension Criminality', 'birth_year': 'Birth Year', 'gender': 'Gender', 'citizenship_country': 'Citizenship Country'})
data2025 = data2025.replace(to_replace="1 Convicted Criminal", value="Convicted")
data2025 = data2025.replace(to_replace="2 Pending Criminal Charges", value="Pending Charges")
data2025 = data2025.replace(to_replace="3 Other Immigration Violator", value="No Criminal Charges")
data2025['aor_nam'] = data2025['aor_nam'].str.replace(' Area of Responsibility', '', regex=False)
data2025['Apprehension MonthYear'] = pd.to_datetime(data2025['Apprehension Date']).dt.strftime('%Y-%m')
data2025['Apprehension Date'] = pd.to_datetime(data2025['Apprehension Date'])
print(data2025['duplicate_likely'].value_counts()) #9455
data2025 = data2025[data2025['duplicate_likely'] == False] #9455 duplicates
data2025['Age'] = data2025['Apprehension Date'].dt.year - data2025['Birth Year'].dropna().astype(int)
data2025['Age Group'] = pd.cut(data2025['Age'], 
                                     bins=[0, 18, 25, 35, 45, 55, 65, 100], 
                                     labels=['0-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+'],
                                     right=False)

#drop weird date
data2025 = data2025[(data2025['Apprehension Date'] <= '2026-04-01')]


duplicate_likely
False    683036
True      16085
Name: count, dtype: int64


In [109]:
fulldata2025 = data2025.copy()
fulldata2025['aor_nam'] = fulldata2025['aor_nam'].fillna('Unknown')
fulldata2025 = fulldata2025.drop(columns=['Apprehension MonthYear', "unique_identifier", "duplicate_likely"])
fulldata2025 = fulldata2025[(fulldata2025['Apprehension Date'] >= '2025-01-20')]
fulldata2025 = fulldata2025[(fulldata2025['Apprehension Date'] <= '2026-04-01')]

fulldata2025.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/individualdatatrump2.csv')

In [110]:
fulldata2025

,Apprehension Date,aor_nam,Apprehension Criminality,Birth Year,Gender,Citizenship Country,Age,Age Group
318097,2025-01-20,Phoenix,Convicted,1988.0,Male,MEXICO,37.0,35-44
318098,2025-01-20,Phoenix,No Criminal Charges,2003.0,Female,VIETNAM,22.0,18-24
318099,2025-01-20,El Paso,No Criminal Charges,1967.0,Male,"CHINA, PEOPLES REPUBLIC OF",58.0,55-64
318100,2025-01-20,Los Angeles,Convicted,1985.0,Male,MEXICO,40.0,35-44
318101,2025-01-20,San Antonio,Convicted,2001.0,Male,HONDURAS,24.0,18-24
...,...,...,...,...,...,...,...,...
713458,2026-03-10,San Antonio,Pending Charges,1980.0,Male,MEXICO,46.0,45-54
713459,2026-03-10,Newark,Pending Charges,1999.0,Male,MEXICO,27.0,25-34
713460,2026-03-10,Houston,Convicted,1980.0,Male,MEXICO,46.0,45-54
713461,2026-03-10,Newark,Convicted,1994.0,Male,MEXICO,32.0,25-34


In [111]:
arrestsbymonth2025 = data2025.groupby('Apprehension MonthYear').size().reset_index(name='arrests')
arrestsbymonth2025.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/arrestsbymonth2025.csv', index=False)

In [112]:
daily_arrests2025 = data2025.groupby('Apprehension Date').size().reset_index(name='arrests')
daily_arrests2025['week_rolling_avg'] = daily_arrests2025['arrests'].rolling(window=7).mean().round()
data2025['date_converted'] = pd.to_datetime(data2025['Apprehension Date'])
daily_arrests2025['Apprehension Date'] = pd.to_datetime(daily_arrests2025['Apprehension Date'])
daily_arrests2025.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/dailyarrests2025.csv', index=False)

In [113]:
data2025

,Apprehension Date,aor_nam,duplicate_likely,unique_identifier,Apprehension Criminality,Birth Year,Gender,Citizenship Country,Apprehension MonthYear,Age,Age Group,date_converted
0,2022-10-01,San Antonio,False,bad3911f91e15e59572fe96a226c1fdbcf82d799,No Criminal Charges,1974.0,Male,CUBA,2022-10,48.0,45-54,2022-10-01
1,2022-10-01,Phoenix,False,dafdb5d0e565d238b9093e99c42e305ac95ce2b4,Convicted,1970.0,Female,PERU,2022-10,52.0,45-54,2022-10-01
2,2022-10-01,Phoenix,False,1db208374a6ce32dd03f1b6d876c9f692b756185,No Criminal Charges,1984.0,Female,COLOMBIA,2022-10,38.0,35-44,2022-10-01
3,2022-10-01,Miami,False,50198f8883c9b118a1153af301c65bfef03bb058,Pending Charges,1996.0,Male,HONDURAS,2022-10,26.0,25-34,2022-10-01
4,2022-10-01,San Antonio,False,84553a21705f50805c41fbdd55c26389e196df2e,No Criminal Charges,1972.0,Female,CUBA,2022-10,50.0,45-54,2022-10-01
...,...,...,...,...,...,...,...,...,...,...,...,...
713458,2026-03-10,San Antonio,False,1eea4bc2de5a11944fc58eb56b180f73270e6f51,Pending Charges,1980.0,Male,MEXICO,2026-03,46.0,45-54,2026-03-10
713459,2026-03-10,Newark,False,b6e64f2f98dd51dfd782be1d3cb170e37faca696,Pending Charges,1999.0,Male,MEXICO,2026-03,27.0,25-34,2026-03-10
713460,2026-03-10,Houston,False,3300c033883216554e0ca8bca139abba21bc05f8,Convicted,1980.0,Male,MEXICO,2026-03,46.0,45-54,2026-03-10
713461,2026-03-10,Newark,False,5a72db4bb17f84699ff794292f9dd0f5d86f12d9,Convicted,1994.0,Male,MEXICO,2026-03,32.0,25-34,2026-03-10


In [114]:
daily_arrests2025aor = data2025.groupby(['aor_nam', 'Apprehension Date']).size().reset_index(name='arrests')
daily_arrests2025aor['week_rolling_avg'] = daily_arrests2025aor['arrests'].rolling(window=7).mean().round()
daily_arrests2025aor['Apprehension Date'] = pd.to_datetime(daily_arrests2025aor['Apprehension Date'])

# Add national aggregate
national_daily = data2025.groupby('Apprehension Date').size().reset_index(name='arrests')
national_daily['aor_nam'] = 'National'
national_daily['week_rolling_avg'] = national_daily['arrests'].rolling(window=7).mean().round()
national_daily['Apprehension Date'] = pd.to_datetime(national_daily['Apprehension Date'])

# Combine regional and national
daily_arrests2025aor = pd.concat([daily_arrests2025aor, national_daily], ignore_index=True)

daily_arrests2025aor.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/dailyarrests2025aor.csv', index=False)

In [115]:
daily_arrests2025aor

,aor_nam,Apprehension Date,arrests,week_rolling_avg
0,Atlanta,2022-10-01,2,NaN
1,Atlanta,2022-10-02,3,NaN
2,Atlanta,2022-10-03,61,NaN
3,Atlanta,2022-10-04,45,NaN
4,Atlanta,2022-10-05,40,NaN
...,...,...,...,...
29475,National,2026-03-07,647,1024.0
29476,National,2026-03-08,500,1010.0
29477,National,2026-03-09,1015,1003.0
29478,National,2026-03-10,1209,994.0


In [116]:
aor_2025 = data2025.groupby(['aor_nam', 'Apprehension MonthYear']).size().reset_index(name='arrests')
aor_2025 = aor_2025[['aor_nam', 'arrests', 'Apprehension MonthYear']]
aor_2025.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/aor_2025_months.csv', index=False)

In [117]:
arrests_by_country2025 = data2025[
    (data2025['Apprehension Date'] >= '2025-01-20')
].groupby('Citizenship Country').agg({
    'unique_identifier': 'count'
}).reset_index()

arrests_by_country2025.columns = ['Citizenship Country', 'Arrests']

# Add percentage
arrests_by_country2025['Percentage'] = (
    arrests_by_country2025['Arrests'] / arrests_by_country2025['Arrests'].sum() * 100
).round(2)

arrests_by_country2025 = arrests_by_country2025.sort_values('Arrests', ascending=False)
arrests_by_country2025["Citizenship Country"] = arrests_by_country2025["Citizenship Country"].str.title()
arrests_by_country2025

,Citizenship Country,Arrests,Percentage
114,Mexico,142420,38.07
75,Guatemala,53371,14.26
80,Honduras,40865,10.92
189,Venezuela,24789,6.63
57,El Salvador,17393,4.65
...,...,...,...
153,Sint Maarten(Dutch),1,0.00
146,Sao Tome And Principe,1,0.00
48,Czechoslovakia,1,0.00
134,Palestine Born Before 1948,1,0.00


In [118]:
topcountries2025 = arrests_by_country2025.nlargest(n=50, columns='Arrests')
topcountries2025
topcountries2025.to_csv('ICEtracker/data/topcountryarrests2025.csv', index=False)

In [119]:
print("The number children in 2025 is", ((data2025['Age'] < 18) & (data2025['Apprehension Date'] >= "2025-01-20")).sum())

The number children in 2025 is 5085


In [147]:
data2025c = pd.read_csv('data/arrests-latest.csv')
columns_to_keep3 = ["apprehension_date", "apprehension_aor", "apprehension_criminality", "unique_identifier"]
data2025c = data2025c[columns_to_keep3]
data2025filtered = data2025c[data2025c['apprehension_date'] >= '2024-10-01'].copy()
data2025filtered
data2025filtered['unique_identifier'].nunique()


396474

In [148]:
import pyarrow.parquet as pq
dettable = pq.read_table("data/detention-stays-latest.parquet")
detdata2025 = dettable.to_pandas()

columns_to_keep2 = [
    "stay_book_in_date_time",
    "most_serious_conviction_code",
    "msc_charge",
    "unique_identifier",
    "citizenship_country"
]


In [149]:

detdata2025 = detdata2025[columns_to_keep2]
detdatafiltered = detdata2025[detdata2025['stay_book_in_date_time'] >= '2024-10-01'].copy()
detdatafiltered['unique_identifier'].nunique()
mergeddata2025 = data2025c.merge(detdata2025, on = "unique_identifier", how = 'right')

In [150]:
print(mergeddata2025['apprehension_aor'].isna().sum())
print(mergeddata2025['stay_book_in_date_time'].isna().sum())

mergeddata2025

502354
0


,apprehension_date,apprehension_aor,apprehension_criminality,unique_identifier,stay_book_in_date_time,most_serious_conviction_code,msc_charge,citizenship_country
0,2025-11-16,San Antonio Area of Responsibility,3 Other Immigration Violator,0000004d7b875782a3b6f2b460f98c0750a5a899,2025-11-16 12:58:00+00:00,None,None,VENEZUELA
1,NaN,NaN,NaN,0000162b33f636fce8b114d00fa8d8645cb76b96,2023-01-05 15:18:00+00:00,None,None,COLOMBIA
2,NaN,NaN,NaN,0000162b33f636fce8b114d00fa8d8645cb76b96,2023-05-17 07:07:00+00:00,None,None,COLOMBIA
3,NaN,NaN,NaN,00001bc128707c1bc0f44bfd6fcc3fa9c3d4f195,2024-12-27 01:16:00+00:00,None,None,MEXICO
4,2025-09-17,Dallas Area of Responsibility,3 Other Immigration Violator,0000215186a4071ae5bc250f63e6baa60b286c30,2025-09-17 10:00:00+00:00,None,None,COLOMBIA
...,...,...,...,...,...,...,...,...
1170927,2025-07-29,New Orleans Area of Responsibility,3 Other Immigration Violator,fffff50ebee0b073b0eb3f5b2ae86920792f48d2,2025-07-29 10:15:00+00:00,None,None,HONDURAS
1170928,2022-11-23,Newark Area of Responsibility,2 Pending Criminal Charges,fffff72ac86811ac3f878d572499191133365990,2022-11-23 19:52:00+00:00,None,None,BRAZIL
1170929,2024-11-08,Newark Area of Responsibility,2 Pending Criminal Charges,fffff72ac86811ac3f878d572499191133365990,2022-11-23 19:52:00+00:00,None,None,BRAZIL
1170930,2022-11-23,Newark Area of Responsibility,2 Pending Criminal Charges,fffff72ac86811ac3f878d572499191133365990,2024-11-08 19:58:00+00:00,None,None,BRAZIL


In [151]:
mergeddata2025 = mergeddata2025[mergeddata2025['apprehension_date'] >= '2024-10-01']
mergeddata2025['apprehension_date'] = pd.to_datetime(mergeddata2025['apprehension_date'])
mergeddata2025 = mergeddata2025.replace(to_replace="1 Convicted Criminal", value="Convicted")
mergeddata2025 = mergeddata2025.replace(to_replace="2 Pending Criminal Charges", value="Pending Charges")
mergeddata2025 = mergeddata2025.replace(to_replace="3 Other Immigration Violator", value="No Criminal Charges")
print(mergeddata2025['apprehension_aor'].isna().sum())
mergeddata2025.drop_duplicates(subset=['unique_identifier', 'apprehension_date', 'apprehension_aor'], inplace=True)
mergeddata2025 = mergeddata2025.dropna(subset=['apprehension_aor'])
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_date': 'Apprehension Date'})
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_aor': 'Apprehension AOR'})
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_criminality': 'Criminality'})
mergeddata2025['Apprehension AOR'] = mergeddata2025['Apprehension AOR'].str.replace(' Area of Responsibility', '', regex=False)

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_47946/1927579702.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mergeddata2025['apprehension_date'] = pd.to_datetime(mergeddata2025['apprehension_date'])


1426


In [152]:
crimelist = pd.read_csv("data/crimeclass.csv")
df = crimelist.rename(columns={'Type of Offense Code  V=violent  D=drug-related  Blank = nonviolent or not drug related': 'Type'})
violentcrimes = df[df['Type'].str.contains('V', na=False)]

In [153]:
mergeddata2025= mergeddata2025.merge(violentcrimes, left_on = "most_serious_conviction_code", right_on = "NCIC Offense Code", how = 'left')
mergeddata2025
mergeddata2025['Type'] = np.where(((mergeddata2025['Type'] != "V") & (mergeddata2025['Criminality'] == "Convicted")), "Convicted - Non-violent", mergeddata2025['Type'])
mergeddata2025['Type'] = np.where((mergeddata2025['Type'] == "V"), "Convicted - Violent crime", mergeddata2025['Type'])
mergeddata2025['Type'] = np.where(((mergeddata2025['Type'] != "V") & (mergeddata2025['Criminality'] == "Pending Charges")), "Pending Charges", mergeddata2025['Type'])
mergeddata2025['Type'] = mergeddata2025['Type'].fillna('No Criminal Charges')

In [154]:
mergeddata2025 = mergeddata2025.drop(columns=['most_serious_conviction_code', 'NCIC Offense Code', 'Description of Crime', 'unique_identifier', 'stay_book_in_date_time', 'msc_charge'])

In [156]:
countriesdf = mergeddata2025.copy()
countriesdf["citizenship_country"] = countriesdf["citizenship_country"].str.title()
countriesdf = countriesdf.drop(columns=['Criminality','Type'])
countrycats = pd.read_csv('ICEtracker/data/countries_by_continents.csv')


In [157]:
countriesdf['citizenship_country'].unique().tolist()

['Venezuela',
 'Colombia',
 'Honduras',
 'Guatemala',
 'Mexico',
 'Brazil',
 'Spain',
 'Peru',
 'Ecuador',
 'Russia',
 'El Salvador',
 'China, Peoples Republic Of',
 'Nicaragua',
 'Haiti',
 'Senegal',
 'Nigeria',
 'India',
 'Cuba',
 'Vietnam',
 'Dominican Republic',
 'Uruguay',
 'Angola',
 'Cameroon',
 'Somalia',
 'Ethiopia',
 'Laos',
 'Turkiye',
 'Argentina',
 'Mauritania',
 'Bangladesh',
 'Italy',
 'Tanzania',
 'Cambodia',
 'Bolivia',
 'Romania',
 'South Korea',
 'Chile',
 'Jamaica',
 'Kosovo',
 'Taiwan',
 'Afghanistan',
 'Uzbekistan',
 'Costa Rica',
 'Iran',
 'Armenia',
 'Bosnia-Herzegovina',
 'Pakistan',
 'Marshall Islands',
 'Kyrgyzstan',
 'Guinea',
 'Micronesia, Federated States Of',
 'Ivory Coast',
 'Morocco',
 'Singapore',
 'Kuwait',
 'South Sudan',
 'Azerbaijan',
 'Burma',
 'Jordan',
 'Poland',
 'Trinidad And Tobago',
 'Equatorial Guinea',
 'Malaysia',
 'Belarus',
 'Korea',
 'Canada',
 'Cape Verde',
 'Iraq',
 'Syria',
 'Ghana',
 'Mali',
 'Tajikistan',
 'Turkmenistan',
 'Egypt'

In [158]:
new_row = pd.DataFrame({'Country': ['Kosovo'], 'Continent': ['Europe']})
countrycats = pd.concat([countrycats, new_row], ignore_index=True)


In [159]:


# Country mapping for ICE data to match reference dataset
# Only includes countries that actually appear in the ICE data

country_mapping = {
    # Formatting differences
    'Antigua-Barbuda': 'Antigua and Barbuda',
    'St. Kitts-Nevis': 'Saint Kitts and Nevis',
    'St. Lucia': 'Saint Lucia',
    'St. Vincent-Grenadines': 'Saint Vincent and the Grenadines',
    'Trinidad And Tobago': 'Trinidad and Tobago',
    'Dem Rep Of The Congo': 'Democratic Republic of Congo',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Sao Tome And Principe': 'Sao Tome and Principe',
    
    # Country name changes
    'Turkiye': 'Turkey',
    'China, Peoples Republic Of': 'China',
    'Burkina Faso': 'Burkina',
    'Czech Republic': 'Czechia',
    'Burma': 'Burma (Myanmar)',  # Reference has "Burma (Myanmar)"
    'North Macedonia': 'Macedonia',
    
    # Historical countries (dissolved) - map to successor states
    'Yugoslavia': 'Serbia',
    'Czechoslovakia': 'Czechia',
    'Ussr': 'Russia',
    'Serbia And Montenegro': 'Serbia',
    
    # Ambiguous/context-dependent
    'Korea': 'North Korea',  # Since 'South Korea' already exists separately in data
    'Macau': 'China',  # Macau is a SAR of China
    # Note: 'Hong Kong' already matches reference data exactly
    
    # Territories - map to parent countries (not in reference as separate entities)
    'French Polynesia': 'France',
    'British Virgin Islands': 'United Kingdom',
    'Turks And Caicos Islands': 'United Kingdom',
    'Cayman Islands': 'United Kingdom',
    'Anguilla': 'United Kingdom',
    'Montserrat': 'United Kingdom',
    'Sint Maarten(Dutch)': 'Netherlands',
    'Micronesia, Federated States Of': 'Micronesia',
}


In [160]:

# Update country_mapping with additional entries
country_mapping.update({
    # Historical countries (dissolved) - map to successor states
    'Yugoslavia': 'Serbia',
    'Czechoslovakia': 'Czechia',
    'Ussr': 'Russia',
    'Serbia And Montenegro': 'Serbia',

    # Ambiguous/context-dependent
    'Korea': 'North Korea',  # Since 'South Korea' already exists separately in data
    'Macau': 'China',  # Macau is a SAR of China
    # Note: 'Hong Kong' already matches reference data exactly

    # Territories - map to parent countries (not in reference as separate entities)
    'French Polynesia': 'France',
    'British Virgin Islands': 'United Kingdom',
    'Turks And Caicos Islands': 'United Kingdom',
    'Cayman Islands': 'United Kingdom',
    'Anguilla': 'United Kingdom',
    'Montserrat': 'United Kingdom',
    'Sint Maarten(Dutch)': 'Netherlands',
    'Micronesia, Federated States Of': 'Micronesia',
})

# Apply the mapping
countriesdf["citizenship_country_cleaned"] = countriesdf["citizenship_country"].replace(country_mapping)

# Merge with reference data
merged = countriesdf.merge(
    countrycats, 
    left_on="citizenship_country_cleaned", 
    right_on="Country", 
    how="left"
)

# Check what didn't merge
unmerged = merged[merged["Continent"].isna()]["citizenship_country_cleaned"].value_counts()

print("=" * 60)
print("MERGE SUMMARY")
print("=" * 60)
print(f"Total records: {len(merged):,}")
print(f"Successfully merged: {(~merged['Continent'].isna()).sum():,} records")
print(f"Unmerged: {merged['Continent'].isna().sum():,} records")
print(f"Merge success rate: {(~merged['Continent'].isna()).sum()/len(merged)*100:.2f}%")

print("\n" + "=" * 60)
print("MAPPINGS APPLIED")
print("=" * 60)
for original, mapped in country_mapping.items():
    count = (countriesdf["citizenship_country"] == original).sum()
    if count > 0:
        print(f"  '{original}' → '{mapped}': {count:,} records")

if len(unmerged) > 0:
    print("\n" + "=" * 60)
    print(f"UNMERGED COUNTRIES ({len(unmerged)} unique)")
    print("=" * 60)
    for country, count in unmerged.items():
        print(f"  {country}: {count:,} records")
    print(f"\nTotal unmerged: {unmerged.sum():,} records ({unmerged.sum()/len(merged)*100:.2f}%)")
else:
    print("\n" + "=" * 60)
    print("✓ ALL COUNTRIES SUCCESSFULLY MERGED!")
    print("=" * 60)

# Display sample of merged data
print("\n" + "=" * 60)
print("SAMPLE OF MERGED DATA")
print("=" * 60)
print(merged[['citizenship_country', 'citizenship_country_cleaned', 'Country', 'Continent']].head(10))

MERGE SUMMARY
Total records: 388,055
Successfully merged: 387,922 records
Unmerged: 133 records
Merge success rate: 99.97%

MAPPINGS APPLIED
  'Antigua-Barbuda' → 'Antigua and Barbuda': 13 records
  'St. Kitts-Nevis' → 'Saint Kitts and Nevis': 16 records
  'St. Lucia' → 'Saint Lucia': 31 records
  'St. Vincent-Grenadines' → 'Saint Vincent and the Grenadines': 11 records
  'Trinidad And Tobago' → 'Trinidad and Tobago': 119 records
  'Dem Rep Of The Congo' → 'Democratic Republic of Congo': 196 records
  'Bosnia-Herzegovina' → 'Bosnia and Herzegovina': 53 records
  'Sao Tome And Principe' → 'Sao Tome and Principe': 1 records
  'Turkiye' → 'Turkey': 994 records
  'China, Peoples Republic Of' → 'China': 3,592 records
  'Burkina Faso' → 'Burkina': 81 records
  'Czech Republic' → 'Czechia': 28 records
  'Burma' → 'Burma (Myanmar)': 187 records
  'North Macedonia' → 'Macedonia': 10 records
  'Yugoslavia' → 'Serbia': 6 records
  'Czechoslovakia' → 'Czechia': 2 records
  'Ussr' → 'Russia': 18 re

In [161]:
merged.to_csv("ICEtracker/data/arrestcountriescontinents.csv")


In [162]:
arrestsbycontinent = merged.groupby(['Apprehension Date', 'Continent']).size().reset_index(name='arrests')
arrestsbycontinent['Apprehension Date'] = pd.to_datetime(arrestsbycontinent['Apprehension Date'])
arrestsbycontinent['week_rolling_avg'] = arrestsbycontinent['arrests'].rolling(window=7).mean().round()
arrestsbycontinent.to_csv('ICEtracker/data/detentioncountscontinents.csv')

In [163]:
arrestsbycontinent

,Apprehension Date,Continent,arrests,week_rolling_avg
0,2024-10-01,Africa,9,NaN
1,2024-10-01,Asia,16,NaN
2,2024-10-01,Europe,2,NaN
3,2024-10-01,North America,282,NaN
4,2024-10-01,Oceania,2,NaN
...,...,...,...,...
2817,2026-03-10,Asia,60,139.0
2818,2026-03-10,Europe,8,138.0
2819,2026-03-10,North America,800,249.0
2820,2026-03-10,South America,161,271.0


In [164]:
# Step 0: Convert to datetime first and remove null values!
mergeddata2025['Apprehension Date'] = pd.to_datetime(mergeddata2025['Apprehension Date'])
mergeddata2025 = mergeddata2025.dropna(subset=['Apprehension Date'])

# Check if mergeddata2025 is empty
if len(mergeddata2025) == 0:
    print("Warning: mergeddata2025 is empty. Using data2025 instead.")
    mergeddata2025 = data2025[['Apprehension Date', 'aor_nam', 'Apprehension Criminality']].copy()
    mergeddata2025 = mergeddata2025.rename(columns={'aor_nam': 'Apprehension AOR', 'Apprehension Criminality': 'Type'})
    mergeddata2025 = mergeddata2025[mergeddata2025['Apprehension Date'] >= '2024-10-01']

# Step 1: Create date range for Trump 2 period (2024-10-01 onwards)
max_date = mergeddata2025['Apprehension Date'].max()
if pd.isna(max_date):
    max_date = pd.Timestamp.now()
    
date_range_2 = pd.date_range(
    start='2024-10-01',
    end=max_date, 
    freq='D'
)

# Step 2: Create combinations for AOR data
from itertools import product

# Trump 2 combinations (from Oct 1, 2024 onwards)
trump2_combinations = pd.DataFrame(
    list(product(
        date_range_2,
        mergeddata2025['Apprehension AOR'].unique(),
        mergeddata2025['Type'].unique()
    )),
    columns=['Apprehension Date', 'Apprehension AOR','Type']
)

# Step 3: Group individual AOR data
chartdf = mergeddata2025.groupby(['Apprehension Date', 'Apprehension AOR', 'Type']).size().reset_index(name='Arrests')

# Step 4: Merge with complete combinations and fill missing with 0
chartdf = trump2_combinations.merge(chartdf, how='left', on=['Apprehension Date', 'Apprehension AOR','Type'])
chartdf['Arrests'] = chartdf['Arrests'].fillna(0)

# Step 5: Create national aggregate with filled dates
trump2_national = pd.DataFrame(
    list(product(
        date_range_2,
        ['Trump 2'],
        mergeddata2025['Type'].unique()
    )),
    columns=['Apprehension Date', 'Administration', 'Type']
)

national_aggregate = mergeddata2025.groupby(['Apprehension Date', 'Type']).size().reset_index(name='Arrests')
national_aggregate['Administration'] = 'Trump 2'

# Merge national aggregate with complete dates
national_aggregate = trump2_national.merge(national_aggregate, how='left', on=['Apprehension Date', 'Administration', 'Type'])
national_aggregate['Arrests'] = national_aggregate['Arrests'].fillna(0)
national_aggregate['Apprehension AOR'] = 'National'

# Step 6: Combine individual AORs with national aggregate
chartdf = pd.concat([chartdf, national_aggregate], ignore_index=True)

# Step 7: Sort properly
chartdf = chartdf.sort_values(by=['Apprehension AOR', 'Type', 'Apprehension Date']).reset_index(drop=True)

# Step 8: Calculate rolling average
chartdf['Week Rolling Average'] = chartdf.groupby(['Apprehension AOR', 'Type'])['Arrests'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
).round(2)

In [165]:
chartdf.to_csv("/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/areachart2025.csv", index=False)

In [166]:
# Create long format dataset with gender, age group, and criminality groupings

# Initialize empty dataframe
data2025v = data2025[data2025["Apprehension Date"] >= "2025-01-20"]
viewdf = pd.DataFrame(columns=['view', 'characteristic', 'value', 'percentage'])
# ===== GENDER GROUPING =====

# Trump 2.0 (2025)
gender_2025 = data2025v.groupby('Gender').size().reset_index(name='value')
gender_2025['view'] = 'gender'
gender_2025['characteristic'] = gender_2025['Gender'].str.lower()
gender_2025['percentage'] = (gender_2025['value'] / gender_2025['value'].sum())
gender_2025 = gender_2025[['view', 'characteristic', 'value', 'percentage']]

viewdf = pd.concat([viewdf, gender_2025], ignore_index=True)

# ===== AGE GROUP GROUPING =====

# Trump 2.0 (2025) - filter to records with valid Age Group
agegroup_2025 = data2025v[data2025v['Age Group'].notna()].groupby('Age Group').size().reset_index(name='value')
agegroup_2025['view'] = 'age group'
agegroup_2025['characteristic'] = agegroup_2025['Age Group'].astype(str)
agegroup_2025['percentage'] = (agegroup_2025['value'] / agegroup_2025['value'].sum())
agegroup_2025 = agegroup_2025[['view', 'characteristic', 'value', 'percentage']]

viewdf = pd.concat([viewdf, agegroup_2025], ignore_index=True)

# ===== CRIMINALITY GROUPING =====
# Trump 2.0 (2025)
criminality_2025 = data2025v.groupby('Apprehension Criminality').size().reset_index(name='value')
criminality_2025['view'] = 'criminality'
criminality_2025['characteristic'] = criminality_2025['Apprehension Criminality'].str.lower()
criminality_2025['percentage'] = (criminality_2025['value'] / criminality_2025['value'].sum())
criminality_2025 = criminality_2025[['view', 'characteristic', 'value', 'percentage']]

viewdf = pd.concat([viewdf, criminality_2025], ignore_index=True)

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_47946/1011801385.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  viewdf = pd.concat([viewdf, gender_2025], ignore_index=True)
/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_47946/1011801385.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agegroup_2025 = data2025v[data2025v['Age Group'].notna()].groupby('Age Group').size().reset_index(name='value')


In [167]:
# Check age group 0-17 in viewdf now
age_017_viewdf = viewdf[(viewdf['view'] == 'age group') & (viewdf['characteristic'] == '0-17')]
print("Age group 0-17 in viewdf (Trump 2.0):", age_017_viewdf['value'].values)

# Check total age groups for Trump 2.0
age_viewdf_2025 = viewdf[(viewdf['view'] == 'age group')]
print("\nAge groups breakdown for Trump 2.0:")
print(age_viewdf_2025[['characteristic', 'value']])
print("\nAge groups total Trump 2.0:", age_viewdf_2025['value'].sum())

Age group 0-17 in viewdf (Trump 2.0): [5085]

Age groups breakdown for Trump 2.0:
  characteristic   value
3           0-17    5085
4          18-24   56114
5          25-34  132445
6          35-44  105173
7          45-54   55396
8          55-64   16722
9            65+    3211

Age groups total Trump 2.0: 374146
